# NexusRAG — 01: Document Ingestion

This notebook demonstrates the ingestion pipeline: parse → chunk → embed → store.

In [ ]:
import sys
sys.path.insert(0, '..')

from src.utils.config import get_settings
settings = get_settings()
print('Settings loaded:', settings.ollama_base_url)

In [ ]:
# Test the document parser
from src.ingestion.parser import DocumentParser

parser = DocumentParser()

# Create a sample text file
import tempfile, os
sample = """# Introduction to Machine Learning

Machine learning is a subset of artificial intelligence.
It enables systems to learn from data.

## Supervised Learning

Supervised learning uses labeled training data.
Examples include classification and regression.
"""
with tempfile.NamedTemporaryFile(mode='w', suffix='.md', delete=False) as f:
    f.write(sample)
    path = f.name

doc = parser.parse(path)
print(f'Type: {doc.doc_type}')
print(f'Content length: {len(doc.content)} chars')
print(doc.content[:200])

In [ ]:
# Test the chunker
from src.ingestion.chunker import SemanticChunker

chunker = SemanticChunker(chunk_size=200, overlap_sentences=1)
chunks = chunker.chunk(doc.content, path)

print(f'Produced {len(chunks)} chunks')
for c in chunks:
    print(f'\n--- Chunk {c.chunk_index}: header={c.header_path!r}')
    print(c.text[:150])

In [ ]:
# Run full ingestion (requires Ollama running)
import asyncio
from src.ingestion.pipeline import IngestionPipeline

pipeline = IngestionPipeline()
result = await pipeline.ingest(path)
print(f'Chunks created: {result.chunks_created}')
print(f'Entities extracted: {result.entities_extracted}')
print(f'Elapsed: {result.elapsed_sec:.2f}s')